# Gold: lab_test_trends

One row per test type per calendar month, built from `silver.lab_results` per the STTM (GOLD/sttm_silver_to_gold.xlsx). Full rebuild with month-over-month abnormal-rate trend via LAG().

In [0]:
# Load type: FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT). Optional LEFT JOIN to calendar on the month's first day.
catalog = "caresyncsydatabricks"

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.lab_test_trends AS
WITH monthly AS (
    SELECT
        test_name,
        year(test_date)                         AS year,
        month(test_date)                        AS month,
        COUNT(*)                                AS total_tests,
        COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')    AS critical_test_count
    FROM {catalog}.silver.lab_results
    GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
    SELECT
        m.*,
        CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
        CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
    FROM monthly m
),
with_lag AS (
    SELECT
        r.*,
        LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
    FROM rates r
)
SELECT
    w.test_name,
    CAST(w.year AS INT)                                                 AS year,
    CAST(w.month AS INT)                                                AS month,
    c.month_name,
    c.quarter_name,
    CAST(w.total_tests AS INT)                                          AS total_tests,
    CAST(w.abnormal_test_count AS INT)                                  AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                                  AS critical_test_count,
    w.abnormal_rate_pct,
    w.critical_rate_pct,
    CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                AS prev_month_abnormal_rate_pct,
    CASE WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
         ELSE CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
    END                                                                 AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {catalog}.silver.calendar c
       ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)
""")

display(spark.table(f"{catalog}.gold.lab_test_trends").orderBy("test_name", "year", "month").limit(20))

In [0]:
# Validation: grain uniqueness (test_name, year, month) and test count reconciliation with silver.lab_results
display(spark.sql(f"""
SELECT
    COUNT(*)                                                    AS gold_rows,
    COUNT(DISTINCT test_name, year, month)                      AS distinct_grain,
    SUM(total_tests)                                            AS total_tests,
    (SELECT COUNT(*) FROM {catalog}.silver.lab_results)         AS silver_lab_results,
    COUNT_IF(month_name IS NULL)                                AS missing_calendar,
    COUNT_IF(prev_month_abnormal_rate_pct IS NULL)              AS first_month_rows,
    COUNT(DISTINCT test_name)                                   AS test_types
FROM {catalog}.gold.lab_test_trends
"""))

In [0]:
record_count = (spark.read.table("caresyncsydatabricks.gold.lab_test_trends").count())

dbutils.notebook.exit(str(record_count))